<a href="https://colab.research.google.com/github/canurdon/SnowLine/blob/main/sentinel2_snow_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import ee
import geemap
import xml.etree.ElementTree as ET
import datetime
import math


ee.Authenticate()
ee.Initialize(project='snowline-491111')


dem = ee.Image('USGS/SRTMGL1_003')
jrc = ee.Image('JRC/GSW1_4/GlobalSurfaceWater')

In [ ]:
!pip install earthengine-api geemap -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 22.4 MB/s eta 0:00:00


In [ ]:

# Snow detection
NDSI_SNOW_THRESHOLD = 0.4       # standard snow/not-snow cutoff
NDSI_STRICT_THRESHOLD = 0.6     # high-confidence snow for snowline derivation
B3_BRIGHTNESS_FLOOR = 1500      # green band reflectance floor — rejects dark NDSI artifacts

# Snowline
SNOWLINE_BUFFER_M = 300         # extend snowline downward by this much (conservative)
SNOWLINE_PERCENTILE = 5        # percentile of confident-snow elevations = snowline

# Staleness
STALENESS_FRESH = 3             # days — pixel considered fresh
STALENESS_MODERATE = 7          # days — pixel considered moderate (>7 = stale)

# Route sampling
ROUTE_BUFFER_M = 30             # corridor width for sampling
MIN_SEGMENT_M = 200             # ignore snow patches shorter than this
MAX_GAP_M = 50                  # merge clear gaps shorter than this

In [ ]:
#create Tantalus bounding box
aoi = ee.Geometry.Rectangle([-123.38, 49.75, -123.10, 49.95])

#Verify area loaded correctly
print('area of interest defined')
print('Bounds:', aoi.bounds().getInfo())

area of interest defined
Bounds: {'geodesic': False, 'type': 'Polygon', 'coordinates': [[[-123.38000000000001, 49.74999999999998], [-123.10000000000001, 49.74999999999998], [-123.10000000000001, 49.95008424772833], [-123.38000000000001, 49.95008424772833], [-123.38000000000001, 49.74999999999998]]]}


##Single Image Snow detection
This section of code applies a snow detection classifier to a single image with low clould cover. The image selected in within a date range contemporaneous with a trip I did into the tantalus. This allows me to assess the classifier based on real world experience of the area during that time.

Small differences in the snow coverage can make a significant difference for athletes travelling through alpine terrain. Therefore, the goal for this section is to produce a snow classifier that us highly accurate and avoids both under and over inclusivity.  

In [ ]:
# Dynamic date range — most recent 60 days

# end = datetime.datetime.now()
# start = end - datetime.timedelta(days=60)

# end_str = end.strftime('%Y-%m-%d')
# start_str = start.strftime('%Y-%m-%d')

start_str= '2024-07-15'
end_str= '2024-07-25'

print(f'Searching: {start_str} to {end_str}')

s2 = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED') \
    .filterBounds(aoi) \
    .filterDate(start_str, end_str) \
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 30)) \
    .sort('system:time_start', False)

image = s2.first()

info = image.getInfo()
print('Image ID:', info['id'])
print('Date:', ee.Date(image.get('system:time_start')).format('YYYY-MM-dd').getInfo())
print('Cloud cover:', image.get('CLOUDY_PIXEL_PERCENTAGE').getInfo(), '%')

Searching: 2024-07-15 to 2024-07-25
Image ID: COPERNICUS/S2_SR_HARMONIZED/20240718T190911_20240718T191829_T10UDA
Date: 2024-07-18
Cloud cover: 0.034934 %


###Basic Snow Mask
This block generates a snow mask using NDSI with a threshold value of 0.3. Because water and water vapour can sometimes produce similar NDSI values to snow, we expect that the classifier will include lakes, rivers, ocean, and cloud as false positives.

In [ ]:

# Compute NDSI
# Formula: (Green - SWIR) / (Green + SWIR)
ndsi = image.normalizedDifference(['B3', 'B11']).rename('NDSI')

# Mask out NDSI below 0.2 (noise from rock/vegetation/soil)
ndsi_display = ndsi.updateMask(ndsi.gt(0))

#Threshhold at 0.3 - standard snow detection cutoff
snow_mask = ndsi.gt(0.3)

# Visualise on an interactive map
map1 = geemap.Map()
map1.centerObject(aoi, zoom=11)

# Add layers
map1.addLayer(
    image,
    {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 3000},
    'True colour'
)
map1.addLayer(
    ndsi,
    {'min': -1, 'max': 1, 'palette': ['black', 'white']},
    'NDSI'
)
map1.addLayer(
    snow_mask,
    {'min': 0, 'max': 1, 'palette': ['000000', '00aaff']},
    'Snow mask'
)
map1.addLayer(
    ndsi_display,
    {'min': 0.1, 'max': 1, 'palette': ['c6eaff', '74b9ff', '2e86de', '0a3d91']},
    'NDSI confidence'
)
map1


Map(center=[49.85001533216063, -123.2400000000015], controls=(WidgetControl(options=['position', 'transparent_…

###Cloud and Water Masks
By masking cloud and water, we can remove some false positives from the NDSI snow mask. EE provides cloud and water classification in its SCL classifier, while water can also be identified using a normalized difference water classifier (NDWI). This block of code masks water and cloud using SCL layer and NDWI. For water, the mask is generated using a combination of NDWI and SCL. SCL layers 3, 8, 9, 10 represent different categories of cloud.  

In [ ]:
# create water and cloud masks

# Get SCL values from image
scl = image.select('SCL')

# Clouds are labelled as scl 3,8,9,10
cloud_mask = scl.neq(3) \
            .And(scl.neq(8)) \
            .And(scl.neq(9)) \
            .And(scl.neq(10))

# Water mask: SCL for obvious water, NDWI for edges
scl_water = scl.eq(6)
ndwi = image.normalizedDifference(['B3', 'B8'])
ndwi_water = ndwi.gt(0.3)
water_mask = scl_water.Not().And(ndwi_water.Not())

# Apply both masks to NDSI
ndsi_masked = ndsi.updateMask(cloud_mask).updateMask(water_mask)

# update NDSI to exclude pixels classified as water and cloud
ndsi_masked =  ndsi.updateMask(water_mask).updateMask(cloud_mask)

# update snow mask
snow_mask_masked = ndsi_masked.gt(0.3)

# update confidence layer
ndsi_display_masked = ndsi_masked.updateMask(ndsi_masked.gt(0))

# show on map

map2 = geemap.Map()
map2.centerObject(aoi, zoom=11)

map2.addLayer(
    image,
    {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 3000},
    'True colour'
    )
map2.addLayer(
    cloud_mask,
     {'min': 0, 'max': 1, 'palette': ['000000', 'ffffff']},
    'Cloud mask'
)
map2.addLayer(
    water_mask,
    {'min': 0, 'max': 1, 'palette': ['000000', 'ffffff']},
    'water mask'
)

map2.addLayer(
    ndsi_display_masked,
    {'min': 0.1, 'max': 1, 'palette': ['c6eaff', '74b9ff', '2e86de', '0a3d91']},
    'NDSI confidence'
    )

map2


Map(center=[49.85001533216063, -123.2400000000015], controls=(WidgetControl(options=['position', 'transparent_…

###Snowline Classifier
While the water mask is fairly effective at removing large bodies of water, it tends to miss patches of water at the edges of bodies of water. If we define the snowline as the elevation above which 95% of high certainty snow is found, then we can use that elevation along with a buffer, to exclude all false positives that occur at elevations where snow is extremely unlikely to occur.

Note: Where glaciers are present, there may be areas of snow that occur well below the snowline. This could lead our snowline condition to remove legitimate areas of perenial snow. This will need to be addressed in later iterations but is not necessary for MVP

In [ ]:
# Derive snowline
strict_snow = ndsi_masked.gt(NDSI_STRICT_THRESHOLD) \
    .And(image.select('B3').gt(B3_BRIGHTNESS_FLOOR))


snowline_elev = dem.updateMask(strict_snow).reduceRegion(
    reducer=ee.Reducer.percentile([SNOWLINE_PERCENTILE]),
    geometry=aoi,
    scale=100,
    maxPixels=1e9
).get('elevation')

# Buffer downward for safety margin (300m)
snowline_buffered = ee.Number(snowline_elev).subtract(SNOWLINE_BUFFER_M)

print('Derived snowline:', snowline_elev.getInfo(), 'm')
print('Buffered snowline:', snowline_buffered.getInfo(), 'm')

# Apply snowline to NDSI, masking everything below buffer
snowline_mask = dem.gte(snowline_buffered)
ndsi_above_snowline = ndsi_display_masked.updateMask(snowline_mask)

map3 = geemap.Map()
map3.centerObject(aoi, zoom=11)

map3.addLayer(
    image,
    {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 3000},
    'True colour'
    )
map3.addLayer(
    strict_snow,
    {'min': 0, 'max': 1, 'palette': ['000000', 'ffffff']},
    'Strict snow'
)
map3.addLayer(
    ndsi_above_snowline,
    {'min': 0.1, 'max': 1, 'palette': ['c6eaff', '74b9ff', '2e86de', '0a3d91']},
    'Confidence above snowline'
)

map3

Derived snowline: 1532.0359168241964 m
Buffered snowline: 1232.0359168241964 m


Map(center=[49.85001533216063, -123.2400000000015], controls=(WidgetControl(options=['position', 'transparent_…

###Compare NDSI snow mask to SCL snow mask
EE uses its own classifier (SCL) to identify snow in each sentinel image. This section of the analysis compares the EE SCL classifier to the NDSI classifier to determine which provides higher accuracy.  


In [ ]:
# SCL-based snow mask — ESA's own classification
# Value 11 = snow/ice in the Scene Classification Layer
scl_full = image.select('SCL')
scl_snow_mask = scl_full.eq(11)

# Compare against NDSI mask
# Four possible outcomes per pixel:
# Both agree snow     = true positive
# Both agree no snow  = true negative
# NDSI says snow, SCL says no = false positive (rock falsely detected)
# SCL says snow, NDSI says no = false negative (snow missed)

true_positive  = snow_mask.And(scl_snow_mask)
true_negative  = snow_mask.Not().And(scl_snow_mask.Not())
false_positive = snow_mask.And(scl_snow_mask.Not())
false_negative = snow_mask.Not().And(scl_snow_mask)

# Count pixels in each category over AOI
def count_pixels(img):
    return img.reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=aoi,
        scale=20,
        maxPixels=1e9
    ).values().get(0)

tp = count_pixels(true_positive).getInfo()
tn = count_pixels(true_negative).getInfo()
fp = count_pixels(false_positive).getInfo()
fn = count_pixels(false_negative).getInfo()

total = tp + tn + fp + fn

print(f'True positives  (both agree snow):     {tp:>8.0f} pixels ({tp/total*100:.1f}%)')
print(f'True negatives  (both agree no snow):  {tn:>8.0f} pixels ({tn/total*100:.1f}%)')
print(f'False positives (NDSI snow, SCL clear):{fp:>8.0f} pixels ({fp/total*100:.1f}%)')
print(f'False negatives (SCL snow, NDSI clear):{fn:>8.0f} pixels ({fn/total*100:.1f}%)')
print()

precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall    = tp / (tp + fn) if (tp + fn) > 0 else 0
accuracy  = (tp + tn) / total

print(f'Precision: {precision:.3f}  (of pixels NDSI calls snow, how many does SCL agree?)')
print(f'Recall:    {recall:.3f}  (of pixels SCL calls snow, how many does NDSI catch?)')
print(f'Accuracy:  {accuracy:.3f}  (overall agreement)')

# Visualise disagreements on map
map5 = geemap.Map()
map5.centerObject(aoi, zoom=11)

map5.addLayer(
    image,
    {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 3000},
    'True colour'
)
map5.addLayer(
    true_positive,
    {'min': 0, 'max': 1, 'palette': ['000000', '0000ff']},
    'True positive (both agree snow)'
)
map5.addLayer(
    false_positive,
    {'min': 0, 'max': 1, 'palette': ['000000', 'ff0000']},
    'False positive (NDSI says snow, SCL disagrees)'
)
map5.addLayer(
    false_negative,
    {'min': 0, 'max': 1, 'palette': ['000000', '00ff00']},
    'False negative (SCL says snow, NDSI misses)'
)
map5.addLayer(
    scl_snow_mask,
    {'min': 0, 'max': 1, 'palette': ['000000', '00ffff']},
    'SCL snow only'
)

map5

True positives  (both agree snow):        62269 pixels (5.6%)
True negatives  (both agree no snow):   1001390 pixels (89.5%)
False positives (NDSI snow, SCL clear):   15768 pixels (1.4%)
False negatives (SCL snow, NDSI clear):   39571 pixels (3.5%)

Precision: 0.798  (of pixels NDSI calls snow, how many does SCL agree?)
Recall:    0.611  (of pixels SCL calls snow, how many does NDSI catch?)
Accuracy:  0.951  (overall agreement)


Map(center=[49.85001533216063, -123.2400000000015], controls=(WidgetControl(options=['position', 'transparent_…

# Multiple Image Compilation
NDSI snow classifiers only work when an image is unobstructed by cloud. Additionally, real-time analysis requires images from recent passes of the sentinel satellite. In summer, when clouds are uncommon, this may not be a problem as a recent cloudless image is likely available. However, in many the winter months, terrain may be covered in cloud for significant periods making a recent clear image impossible to find.

To solve this problem, we can use a compilation of images to obtain a complete set of cloud-free pixels for the AOI. We can then use this to derive NDSI for the area of interest. The result is the most recent cloud-free NDSI value for every pixel in the AOI, giving the user the best possible estimate for that pixel. By recording the timestamp for each pixel, we can also identify the staleness of each pixels NDSI and build this into the classifier as a variable affecting confidence.


###Process Imgaes
We gather a collection of images for the previous 60 days (a period that must be long enough to ensure a cloud-free image over each pixel). We then process each image to derive its masked-NDSI—the NDSI of clear pixels in that image—as well as the images timestamp and B3. (true colour bands could also be added for visual verification, but should not be returned in final product to save compute)

In [ ]:
# collect a collection of images for the last 60 days

end_date = ee.Date(datetime.datetime.now())
start_date = end_date.advance(-60, 'day')

collection = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED') \
    .filterBounds(aoi) \
    .filterDate(start_date, end_date)

count = collection.size().getInfo()
dates = collection.aggregate_array('system:time_start').getInfo()
clouds = collection.aggregate_array('CLOUDY_PIXEL_PERCENTAGE').getInfo()

print(f'Images in collection: {count}\n')
for d, c in zip(dates, clouds):
    date_str = ee.Date(d).format('YYYY-MM-dd').getInfo()
    print(f'  {date_str} | {c:.1f}% cloud')

# Create a function that takes an image, and

def process_image(image):
    """Per-image processing: mask clouds/water, compute NDSI, track timestamp.

    Only the continuous NDSI value flows through the composite — no binary
    thresholding here. This preserves the full signal so post-composite
    classification can apply the snowline gate and produce confidence levels,
    not just snow/not-snow.
    """
    scl = image.select('SCL')

    # Cloud mask: SCL 3=shadow, 8=cloud medium, 9=cloud high, 10=cirrus
    cloud_mask = scl.neq(3).And(scl.neq(8)).And(scl.neq(9)).And(scl.neq(10))

    # Water mask: SCL class 6 for obvious bodies, NDWI for edges and streams
    # Both are per-image so frozen snow-covered lakes pass through correctly
    scl_water = scl.eq(6)
    ndwi = image.normalizedDifference(['B3', 'B8'])
    ndwi_water = ndwi.gt(0.3)
    water_mask = scl_water.Not().And(ndwi_water.Not())

    # Combined mask: pixel must pass both cloud AND water checks
    valid_mask = cloud_mask.And(water_mask)

    # NDSI: continuous value (-1 to 1), higher = more likely snow
    ndsi = image.normalizedDifference(['B3', 'B11']).rename('NDSI')
    ndsi_masked = ndsi.updateMask(valid_mask)

    # True colour (can be removed after testing)
    true_colour = image.select(['B4', 'B3', 'B2'])
    true_colour_masked = true_colour.updateMask(valid_mask)

    # Carry B3 for post-composite snowline derivation
    b3_masked = image.select('B3').updateMask(valid_mask)

    # Timestamp: when this pixel was observed
    # Must be masked identically to NDSI — otherwise qualityMosaic picks
    # the newest timestamp even where NDSI is null (clouded out),
    # resulting in empty pixels claiming to be from the most recent image
    timestamp = image.metadata('system:time_start').rename('timestamp').clip(aoi)
    timestamp_masked = timestamp.updateMask(valid_mask)

    return ndsi_masked \
        .addBands(true_colour_masked) \
        .addBands(b3_masked) \
        .addBands(timestamp_masked) \
        .clip(aoi) \
        .copyProperties(image, ['system:time_start'])


processed = collection.map(process_image)
composite = processed.qualityMosaic('timestamp')
#.clip(aoi)


print('Composite bands:', composite.bandNames().getInfo())
# Expected: ['NDSI', 'timestamp']


Images in collection: 27

  2026-02-08 | 31.3% cloud
  2026-02-11 | 29.1% cloud
  2026-02-13 | 89.4% cloud
  2026-02-16 | 99.0% cloud
  2026-02-18 | 93.9% cloud
  2026-02-21 | 99.9% cloud
  2026-02-23 | 80.8% cloud
  2026-02-26 | 98.7% cloud
  2026-02-28 | 55.9% cloud
  2026-03-03 | 99.9% cloud
  2026-03-05 | 30.5% cloud
  2026-03-08 | 29.0% cloud
  2026-03-10 | 99.9% cloud
  2026-03-13 | 35.6% cloud
  2026-03-15 | 99.4% cloud
  2026-03-15 | 99.6% cloud
  2026-03-18 | 99.8% cloud
  2026-03-20 | 84.8% cloud
  2026-03-25 | 97.6% cloud
  2026-03-28 | 79.8% cloud
  2026-03-30 | 15.3% cloud
  2026-04-01 | 96.5% cloud
  2026-04-02 | 78.5% cloud
  2026-04-02 | 80.5% cloud
  2026-04-04 | 33.5% cloud
  2026-04-04 | 31.8% cloud
  2026-04-07 | 12.4% cloud
Composite bands: ['NDSI', 'B4', 'B3', 'B2', 'B3_1', 'timestamp']


###Pixel Staleness image
Now we can visualize the timestamp associated with each pixel. In areas where the most recent images included cloud, we should see patches of colour that represent pixels whos values are associated with older images.

We can also print the dates of the images used to create the mosaic image, although the resampling of the image means that some of these may be errors. This could be fixed in later version by avoding resampling.


In [ ]:
# The timestamp band already tells you which image each pixel came from
pixel_dates = composite.select('timestamp')
NDSI = composite.select('NDSI')
NDSI_masked = NDSI.updateMask(NDSI.gt(0))

# Visualise with a discrete palette — each colour = a different source date
map_sources = geemap.Map()
map_sources.centerObject(aoi, zoom=11)

map_sources.addLayer(
    NDSI_masked,
    {'min': 0.1, 'max': 1, 'palette': ['c6eaff', '74b9ff', '2e86de', '0a3d91']},
    'Confidence Snow'
)

map_sources.addLayer(
    pixel_dates,
    {
        'min': ee.Date(start_date).millis().getInfo(),
        'max': ee.Date(end_date).millis().getInfo(),
        'palette': ['ff0000', 'ff8800', 'ffff00', '00ff00', '0088ff', '8800ff']
    },
    'Pixel source date (red=oldest, purple=newest)'
)

map_sources


Map(center=[49.85001533216063, -123.2400000000015], controls=(WidgetControl(options=['position', 'transparent_…

In [ ]:
ts_daily = composite.select('timestamp') \
    .divide(86400000).floor().multiply(86400000) \
    .rename('timestamp')

sample_dates = ts_daily.reproject(
    crs='EPSG:4326', scale=100
).reduceRegion(
    reducer=ee.Reducer.frequencyHistogram(),
    geometry=aoi, scale=100, maxPixels=1e9
).get('timestamp')

hist = sample_dates.getInfo()
print(f'Unique observation dates in composite: {len(hist)}\n')

for ts_str, pixel_count in sorted(hist.items(), key=lambda x: -x[1]):
    if pixel_count < 1:
        continue
    date = ee.Date(float(ts_str)).format('YYYY-MM-dd').getInfo()
    print(f'  {date} | {int(pixel_count)} pixels')

Unique observation dates in composite: 37

  2026-04-07 | 58542 pixels
  2026-04-04 | 7052 pixels
  2026-03-30 | 2460 pixels
  2026-03-28 | 253 pixels
  2026-03-13 | 239 pixels
  2026-04-02 | 165 pixels
  2026-04-06 | 80 pixels
  2026-03-08 | 52 pixels
  2026-04-05 | 47 pixels
  2026-04-03 | 44 pixels
  2026-02-11 | 30 pixels
  2026-03-31 | 21 pixels
  2026-03-23 | 19 pixels
  2026-02-28 | 17 pixels
  2026-03-05 | 15 pixels
  2026-02-23 | 12 pixels
  2026-03-25 | 9 pixels
  2026-03-24 | 7 pixels
  2026-03-20 | 7 pixels
  2026-03-26 | 7 pixels
  2026-03-21 | 6 pixels
  2026-03-18 | 5 pixels
  2026-03-09 | 4 pixels
  2026-04-01 | 3 pixels
  2026-03-22 | 2 pixels
  2026-03-27 | 1 pixels
  2026-03-16 | 1 pixels
  2026-03-17 | 1 pixels
  2026-03-04 | 1 pixels
  2026-02-08 | 1 pixels


## On which days were the images used to make the compilation taken?

In [ ]:
# Round timestamps to nearest day to eliminate interpolation artifacts
ts_daily = composite.select('timestamp') \
    .divide(86400000).floor().multiply(86400000) \
    .rename('timestamp')

sample_dates = ts_daily.reproject(
    crs='EPSG:4326', scale=100
).reduceRegion(
    reducer=ee.Reducer.frequencyHistogram(),
    geometry=aoi,
    scale=100,
    maxPixels=1e9
).get('timestamp')

hist = sample_dates.getInfo()

# Get actual collection dates for filtering out interpolation artifacts
collection_timestamps = collection.aggregate_array('system:time_start').getInfo()
collection_days = set()
for ts in collection_timestamps:
    day_ts = int(ts // 86400000) * 86400000
    collection_days.add(day_ts)

# Print only dates that exist in the collection
real_dates = 0
for ts_str, pixel_count in sorted(hist.items(), key=lambda x: -x[1]):
    ts_day = int(float(ts_str) // 86400000) * 86400000
    if ts_day not in collection_days:
        continue
    real_dates += 1
    date = ee.Date(float(ts_str)).format('YYYY-MM-dd').getInfo()
    print(f'  {date} | {int(pixel_count)} pixels')

print(f'\n{real_dates} real dates contributing to composite')

  2026-04-07 | 58542 pixels
  2026-04-04 | 7052 pixels
  2026-03-30 | 2460 pixels
  2026-03-28 | 253 pixels
  2026-03-13 | 239 pixels
  2026-04-02 | 165 pixels
  2026-03-08 | 52 pixels
  2026-02-11 | 30 pixels
  2026-02-28 | 17 pixels
  2026-03-05 | 15 pixels
  2026-02-23 | 12 pixels
  2026-03-25 | 9 pixels
  2026-03-20 | 7 pixels
  2026-03-18 | 5 pixels
  2026-04-01 | 3 pixels
  2026-02-08 | 1 pixels
  2026-03-15 | 0 pixels
  2026-03-10 | 0 pixels

18 real dates contributing to composite


In [ ]:
# This is the key step: derive the snowline from the composite itself,
# then re-classify using the snowline as a dynamic elevation gate.

ndsi_composite = composite.select('NDSI')

# Step 1: Find "definitely snow" pixels in the composite
# Strict NDSI + brightness floor = high-confidence snow
# We need the B3 brightness from a recent clear image for this.
# For Use the most recent low-cloud image from the collection.
recent_clear = collection \
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 30)) \
    .sort('system:time_start', False) \
    .first()

# Cloud mask for the reference image
ref_scl = recent_clear.select('SCL')
ref_cloud_mask = ref_scl.neq(3).And(ref_scl.neq(8)).And(ref_scl.neq(9)).And(ref_scl.neq(10))
ref_water_mask = ref_scl.neq(6)
ref_ndsi = recent_clear.normalizedDifference(['B3', 'B11'])

#strict Snow
strict_snow = ndsi_composite.gt(NDSI_STRICT_THRESHOLD) \
    .And(composite.select('B3').gt(B3_BRIGHTNESS_FLOOR))

# Step 2: Derive snowline = 5th percentile elevation of confident snow pixels
snowline_elev = dem.updateMask(strict_snow).reduceRegion(
    reducer=ee.Reducer.percentile([SNOWLINE_PERCENTILE]),
    geometry=aoi,
    scale=100,
    maxPixels=1e9
).get('elevation')

# Buffer downward for safety margin
snowline_buffered = ee.Number(snowline_elev).subtract(SNOWLINE_BUFFER_M)

print('Derived snowline:', snowline_elev.getInfo(), 'm')
print('Buffered snowline:', snowline_buffered.getInfo(), 'm')

# Step 3: Apply snowline to composite
above_snowline = dem.gte(snowline_buffered)
snow_binary = ndsi_composite.gt(NDSI_SNOW_THRESHOLD)
snow_confident = snow_binary.And(above_snowline)

# Step 4: Staleness
now_ms = ee.Date(datetime.datetime.now()).millis()
days_ago = composite.select('timestamp') \
    .subtract(now_ms).abs().divide(86400000).rename('days_ago')

# Step 5: Staleness classification band
staleness_class = days_ago.where(days_ago.lte(STALENESS_FRESH), 1) \
    .where(days_ago.gt(STALENESS_FRESH).And(days_ago.lte(STALENESS_MODERATE)), 2) \
    .where(days_ago.gt(STALENESS_MODERATE), 3) \
    .rename('staleness')

# ── Package final output ─────────────────────────────────────────────────
# snow = binary gate (NDSI > threshold AND above snowline)
# NDSI = continuous value — drives confidence display in the app
# above_snowline = elevation gate
# days_ago = pixel staleness in days
# staleness = classified staleness (1=fresh, 2=moderate, 3=stale)
output = snow_confident.rename('snow') \
    .addBands(ndsi_composite) \
    .addBands(above_snowline.rename('above_snowline')) \
    .addBands(days_ago) \
    .addBands(staleness_class)

print('\nFinal output bands:', output.bandNames().getInfo())
# Expected: ['snow', 'NDSI', 'above_snowline', 'days_ago', 'staleness']

Derived snowline: 1003.2477776064952 m
Buffered snowline: 703.2477776064952 m

Final output bands: ['snow', 'NDSI', 'above_snowline', 'days_ago', 'staleness']


In [ ]:
ndsi_composite = composite.select('NDSI')
b3_composite = composite.select('B3')

# Derive snowline from the composite itself
# Every pixel's NDSI and B3 come from the same observation (qualityMosaic guarantees this)
strict_snow = ndsi_composite.gt(NDSI_STRICT_THRESHOLD) \
    .And(b3_composite.gt(B3_BRIGHTNESS_FLOOR))

snowline_elev = dem.updateMask(strict_snow).reduceRegion(
    reducer=ee.Reducer.percentile([SNOWLINE_PERCENTILE]),
    geometry=aoi, scale=100, maxPixels=1e9
).get('elevation')

# Buffer downward for safety margin
snowline = ee.Number(snowline_elev)
snowline_buffered = ee.Number(snowline_elev).subtract(SNOWLINE_BUFFER_M)

# Generate Snow Confidence Mask as NDSI above Duffered Snowline
snowline_mask = dem.gte(snowline_buffered)
ndsi_masked = ndsi_composite.updateMask(ndsi_composite.gt(0.1))

ndsi_above_snowline = ndsi_masked.updateMask(snowline_mask)

print('Derived snowline:', snowline_elev.getInfo(), 'm')
print('Buffered snowline:', snowline_buffered.getInfo(), 'm')

Derived snowline: 1003.2477776064952 m
Buffered snowline: 703.2477776064952 m


In [ ]:
map3 = geemap.Map()
map3.centerObject(aoi, zoom=11)

# True colour from most recent clear image for context
map3.addLayer(
    composite,
    {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 3000},
    'True colour (composite)'
)

# Snow: final confident classification
map3.addLayer(
    strict_snow,
    {'min': 0, 'max': 1, 'palette': ['000000', '00aaff']},
    'Snow (confident)'
)

# # NDSI intensity gradient
# map3.addLayer(
#     output.select('NDSI'),
#     {'min': 0.2, 'max': 1, 'palette': ['2c1654', '4a90d9', '00aaff', 'e0f7ff']},
#     'NDSI intensity'
# )

# Generate countours for snowline and snowline buffer
snowline_contour = dem.gte(snowline).And(dem.lt(snowline.add(50)))
snowline_buffer_contour = dem.gte(snowline_buffered).And(dem.lt(snowline_buffered.add(50)))

# Snowline buffer contour — show where the dynamic cutoff sits
map3.addLayer(
    snowline_buffer_contour.selfMask(),
    {'palette': ['ff6600']},
    'Snowlinebuffer contour'
)
# Snowline contour
map3.addLayer(
    snowline_contour.selfMask(),
    {'palette': ['ff00ff']},
    'Snowline contour'
)
# Pixels below snowline that NDSI thinks are snow — the ones we're rejecting
rejected = strict_snow.And(snowline_buffered.Not())
map3.addLayer(
    rejected.selfMask(),
    {'palette': ['ff0000']},
    'Rejected (below snowline)'
)

# # Staleness
# map3.addLayer(
#     output.select('days_ago'),
#     {'min': 0, 'max': 60, 'palette': ['00ff00', 'ffff00', 'ff0000']},
#     'Pixel age (days)'
# )

# Elevation
map3.addLayer(
    dem,
    {'min': 0, 'max': 2600, 'palette': ['000000', '555555', 'aaaaaa', 'ffffff']},
    'Elevation (SRTM)'
)
# Snow above Snowline
map3.addLayer(
    ndsi_above_snowline,
    {'min': 0.1, 'max': 1, 'palette': ['c6eaff', '74b9ff', '2e86de', '0a3d91']},
    'Confidence above snowline'
)


map3


Map(center=[49.85001533216063, -123.2400000000015], controls=(WidgetControl(options=['position', 'transparent_…

# GPS Track Functionality
These cells allow the user to upload a GPS track and analyse the route against the terrain layers.

In [ ]:
# Parse GPX file
def parse_gpx(filepath):
    tree = ET.parse(filepath)
    root = tree.getroot()

    # Handle GPX namespace
    ns = {'gpx': 'http://www.topografix.com/GPX/1/1'}

    coords = []
    elevations = []

    for trkpt in root.findall('.//gpx:trkpt', ns):
        lat = float(trkpt.get('lat'))
        lon = float(trkpt.get('lon'))
        ele = trkpt.find('gpx:ele', ns)
        elev = float(ele.text) if ele is not None else 0

        coords.append([lon, lat])
        elevations.append(elev)

    return coords, elevations

# Load the route
coords, elevations = parse_gpx('/content/drive/Othercomputers/My MacBook Pro (1)/Desktop/SnowLine/data/tantalusFKT_2024.gpx')

print(f'Track points: {len(coords)}')
print(f'Start: {coords[0]}')
print(f'End: {coords[-1]}')
print(f'Min elevation: {min(elevations)}m')
print(f'Max elevation: {max(elevations)}m')

Track points: 21699
Start: [-123.3229366, 49.9105733]
End: [-123.2026349, 49.79482]
Min elevation: 101.0m
Max elevation: 2601.0m


In [ ]:
# Simplify route — take every 50th point
# 21699 / 50 = ~434 points, plenty for 10m resolution
simplified = coords[::10]
print(f'Simplified to {len(simplified)} points')

# Create Earth Engine line geometry
route = ee.Geometry.LineString(simplified)

# Display on map with new AOI
map4 = geemap.Map()
map4.centerObject(route, zoom=11)

# True colour from most recent clear image for context
map4.addLayer(
    composite,
    {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 3000},
    'True colour (composite)'
)

# Add snow composite
map4.addLayer(
    strict_snow,
    {'min': 0, 'max': 1, 'palette': ['000000', '00aaff']},
    'Snow (confident)'
)

# Add route
map4.addLayer(
    ee.Image().paint(route, 1, 2),
    {'palette': ['ff0000']},
    'Tantalus Traverse'
)

map4

Simplified to 2170 points


Map(center=[49.837624264988584, -123.31348506118746], controls=(WidgetControl(options=['position', 'transparen…

In [ ]:
route_buffer = route.buffer(ROUTE_BUFFER_M)

# Sample from the final output — all bands at once
sampled = composite.select(['snow', 'NDSI', 'above_snowline', 'days_ago']) \
    .sample(
        region=route_buffer,
        scale=10,
        numPixels=500,
        geometries=True
    )

# Counts
total = sampled.size().getInfo()
snow_count = sampled.filter(ee.Filter.eq('snow', 1)).size().getInfo()
clear_count = total - snow_count

print(f'Total sampled points: {total}')
print(f'Snow points: {snow_count}')
print(f'Clear points: {clear_count}')
print(f'Snow coverage: {round(snow_count/total*100, 1)}%')

# Detailed stats
features = sampled.getInfo()['features']

ndsi_values = [f['properties']['NDSI'] for f in features
               if f['properties']['snow'] == 1 and f['properties']['NDSI'] is not None]
below_snowline_hits = sum(1 for f in features
                          if f['properties'].get('above_snowline') == 0
                          and f['properties']['NDSI'] is not None
                          and f['properties']['NDSI'] > NDSI_SNOW_THRESHOLD)

if ndsi_values:
    print(f'\nSnow pixel NDSI — mean: {sum(ndsi_values)/len(ndsi_values):.3f}, '
          f'min: {min(ndsi_values):.3f}, max: {max(ndsi_values):.3f}')
    print(f'Rejected by snowline: {below_snowline_hits} pixels had NDSI > {NDSI_SNOW_THRESHOLD} '
          f'but were below the derived snowline')


Total sampled points: 500
Snow points: 312
Clear points: 188
Snow coverage: 62.4%

Snow pixel NDSI — mean: 0.896, min: 0.422, max: 0.984
Rejected by snowline: 0 pixels had NDSI > 0.4 but were below the derived snowline


In [ ]:
points = []
for f in features:
    coords = f['geometry']['coordinates']
    props = f['properties']

    points.append({
        'lon': coords[0],
        'lat': coords[1],
        'snow': props['snow'],
        'ndsi': props.get('NDSI'),
        'above_snowline': props.get('above_snowline'),
        'days_ago': round(props.get('days_ago'), 1) if props.get('days_ago') is not None else None,
    })

print(f'Extracted {len(points)} points')
print(f'Sample: {points[0]}')

Extracted 500 points
Sample: {'lon': -123.29261115668555, 'lat': 49.788460958322524, 'snow': 1, 'ndsi': 0.8023015856742859, 'above_snowline': 1, 'days_ago': 1.0}


In [ ]:
def haversine(coord1, coord2):
    """Distance in metres between two [lon, lat] points"""
    R = 6371000
    lat1, lat2 = math.radians(coord1[1]), math.radians(coord2[1])
    dlat = math.radians(coord2[1] - coord1[1])
    dlon = math.radians(coord2[0] - coord1[0])
    a = math.sin(dlat/2)**2 + math.cos(lat1)*math.cos(lat2)*math.sin(dlon/2)**2
    return R * 2 * math.asin(math.sqrt(a))

# Build list of points with cumulative distance along route
# Use original simplified coords as the route backbone
route_with_dist = []
cumulative_dist = 0
for i, coord in enumerate(simplified):
    if i > 0:
        cumulative_dist += haversine(simplified[i-1], coord)
    route_with_dist.append({
        'lon': coord[0],
        'lat': coord[1],
        'dist_m': cumulative_dist
    })

total_route_km = cumulative_dist / 1000
print(f'Total route length: {total_route_km:.1f} km')

# For each sampled point, find nearest route point and get its distance
def nearest_route_dist(point, route):
    min_dist = float('inf')
    nearest = None
    for rp in route:
        d = haversine([point['lon'], point['lat']], [rp['lon'], rp['lat']])
        if d < min_dist:
            min_dist = d
            nearest = rp
    return nearest['dist_m']

# Tag each sample point with distance along route
print('Calculating distances along route...')
for p in points:
    p['route_dist_m'] = nearest_route_dist(p, route_with_dist)

# Sort by distance along route
points.sort(key=lambda x: x['route_dist_m'])
print('Done')


Total route length: 29.5 km
Calculating distances along route...
Done


In [ ]:
segments_raw = []
current_state = points[0]['snow']
segment_start = points[0]
segment_points = [points[0]]

for p in points[1:]:
    if p['snow'] != current_state:
        dist_m = segment_points[-1]['route_dist_m'] - segment_start['route_dist_m']
        ndsi_vals = [pt['ndsi'] for pt in segment_points if pt['ndsi'] is not None]
        days_vals = [pt['days_ago'] for pt in segment_points if pt['days_ago'] is not None]

        segments_raw.append({
            'type': 'snow' if current_state == 1 else 'clear',
            'start_m': segment_start['route_dist_m'],
            'end_m': segment_points[-1]['route_dist_m'],
            'length_m': dist_m,
            'points': segment_points,
            'mean_ndsi': sum(ndsi_vals) / max(1, len(ndsi_vals)),
            'max_days_ago': max(days_vals) if days_vals else None,
        })
        current_state = p['snow']
        segment_start = p
        segment_points = [p]
    else:
        segment_points.append(p)

# Last segment
ndsi_vals = [pt['ndsi'] for pt in segment_points if pt['ndsi'] is not None]
days_vals = [pt['days_ago'] for pt in segment_points if pt['days_ago'] is not None]
segments_raw.append({
    'type': 'snow' if current_state == 1 else 'clear',
    'start_m': segment_start['route_dist_m'],
    'end_m': segment_points[-1]['route_dist_m'],
    'length_m': segment_points[-1]['route_dist_m'] - segment_start['route_dist_m'],
    'points': segment_points,
    'mean_ndsi': sum(ndsi_vals) / max(1, len(ndsi_vals)),
    'max_days_ago': max(days_vals) if days_vals else None,
})

# Gap merging
segments_merged = [segments_raw[0]]
for seg in segments_raw[1:]:
    prev = segments_merged[-1]
    if (prev['type'] == 'clear'
            and prev['length_m'] < MAX_GAP_M
            and len(segments_merged) >= 2
            and segments_merged[-2]['type'] == 'snow'
            and seg['type'] == 'snow'):
        snow_before = segments_merged[-2]
        all_points = snow_before['points'] + prev['points'] + seg['points']
        ndsi_all = [pt['ndsi'] for pt in all_points if pt['ndsi'] is not None]
        days_all = [pt['days_ago'] for pt in all_points if pt['days_ago'] is not None]
        snow_before['end_m'] = seg['end_m']
        snow_before['length_m'] = seg['end_m'] - snow_before['start_m']
        snow_before['points'] = all_points
        snow_before['mean_ndsi'] = sum(ndsi_all) / max(1, len(ndsi_all))
        snow_before['max_days_ago'] = max(days_all) if days_all else None
        segments_merged.pop()
    else:
        segments_merged.append(seg)

# Length filter
segments_final = [s for s in segments_merged if s['length_m'] >= MIN_SEGMENT_M]

# Report
snow_segs = [s for s in segments_final if s['type'] == 'snow']
print(f'\nSnow crossings: {len(snow_segs)}')
print(f'({len(segments_raw)} raw → {len(segments_merged)} merged → {len(segments_final)} final)\n')

for i, seg in enumerate(snow_segs):
    staleness = f"{seg['max_days_ago']:.0f}d old" if seg['max_days_ago'] else "age unknown"
    ndsi_str = f"NDSI {seg['mean_ndsi']:.2f}"
    confidence = "strong" if seg['mean_ndsi'] > 0.6 else "moderate" if seg['mean_ndsi'] > 0.45 else "weak"

    print(f"  Crossing {i+1}: {seg['start_m']/1000:.1f}–{seg['end_m']/1000:.1f} km "
          f"({seg['length_m']:.0f}m) | {ndsi_str} ({confidence}) | {staleness}")



Snow crossings: 3
(35 raw → 17 merged → 10 final)

  Crossing 1: 5.8–6.4 km (621m) | NDSI 0.57 (moderate) | 4d old
  Crossing 2: 7.2–23.1 km (15879m) | NDSI 0.92 (strong) | 4d old
  Crossing 3: 23.5–25.4 km (1930m) | NDSI 0.63 (strong) | 4d old
